In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

file_paths = [
    r"E:\ML Project\WOSC Project\EarthFormer\Data\amphan_2020-05-06_to_2020-06-04.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Data\asani_2022-04-27_to_2022-05-26.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Data\dana_2024-10-12_to_2024-11-10.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Data\fani_2019-04-16_to_2019-05-15.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Data\gulab_2021-09-14_to_2021-10-13.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Data\hudhud_2014-09-27_to_2014-10-26.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Data\phailin_2013-09-24_to_2013-10-23.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Data\titli_2018-09-28_to_2018-10-27.nc",
    r"E:\ML Project\WOSC Project\EarthFormer\Data\yaas_2021-05-13_to_2021-06-11.nc" 
]

datasets = []

for path in file_paths:
    ds = xr.open_dataset(path)
    datasets.append(ds)

ds.data_vars

Data variables:
    to       (time, depth, latitude, longitude) float64 29MB ...
    so       (time, depth, latitude, longitude) float64 29MB ...
    ugo      (time, depth, latitude, longitude) float64 29MB ...
    vgo      (time, depth, latitude, longitude) float64 29MB ...
    zo       (time, depth, latitude, longitude) float64 29MB ...

### 1. Defining a Common Ocean Mask

In [2]:
import numpy as np

cleaned_datasets = []

for i, ds in enumerate(datasets):
    print(f"Cleaning for Cyclone {i+1}")

    # Create a common NaN mask across all variables
    combined_mask = None
    for v in ds.data_vars:
        vmask = np.isnan(ds[v])
        if combined_mask is None:
            combined_mask = vmask
        else:
            combined_mask = combined_mask | vmask  # union of NaNs

    # Apply the mask to all variables
    for v in ds.data_vars:
        ds[v] = ds[v].where(~combined_mask)

    # Append cleaned dataset to the list
    cleaned_datasets.append(ds)

    # Quick stats to check cleaning
    print("NaN fraction per variable after cleaning:")
    for v in ds.data_vars:
        nan_frac = np.isnan(ds[v]).mean().item()
        print(f"  {v}: {nan_frac:.4f}")
    print("---------------------------------------------------")

for i, ds in enumerate(cleaned_datasets):
    print(f"\nMask check Cyclone {i+1}:")
    mask_ref = np.isnan(ds[list(ds.data_vars)[0]])  # reference variable
    for v in ds.data_vars:
        stable = np.all(mask_ref == np.isnan(ds[v]))
        print(f"{v} mask same as reference? {stable}")


Cleaning for Cyclone 1
NaN fraction per variable after cleaning:
  to: 0.4300
  so: 0.4300
  ugo: 0.4300
  vgo: 0.4300
  zo: 0.4300
---------------------------------------------------
Cleaning for Cyclone 2
NaN fraction per variable after cleaning:
  to: 0.4300
  so: 0.4300
  ugo: 0.4300
  vgo: 0.4300
  zo: 0.4300
---------------------------------------------------
Cleaning for Cyclone 3
NaN fraction per variable after cleaning:
  to: 0.4300
  so: 0.4300
  ugo: 0.4300
  vgo: 0.4300
  zo: 0.4300
---------------------------------------------------
Cleaning for Cyclone 4
NaN fraction per variable after cleaning:
  to: 0.4300
  so: 0.4300
  ugo: 0.4300
  vgo: 0.4300
  zo: 0.4300
---------------------------------------------------
Cleaning for Cyclone 5
NaN fraction per variable after cleaning:
  to: 0.4300
  so: 0.4300
  ugo: 0.4300
  vgo: 0.4300
  zo: 0.4300
---------------------------------------------------
Cleaning for Cyclone 6
NaN fraction per variable after cleaning:
  to: 0.4300
  

### 2. Spatial Domain and Depth Check

In [3]:
lat_ref = cleaned_datasets[0]["latitude"]
lon_ref = cleaned_datasets[0]["longitude"]

for i, ds in enumerate(cleaned_datasets[1:], start=2):
    assert np.array_equal(ds["latitude"], lat_ref), f"Latitude mismatch in Cyclone {i}"
    assert np.array_equal(ds["longitude"], lon_ref), f"Longitude mismatch in Cyclone {i}"

print("All datasets have consistent spatial domain")

depth_ref = cleaned_datasets[0]["depth"]

for i, ds in enumerate(cleaned_datasets[1:], start=2):
    assert np.array_equal(ds["depth"], depth_ref), f"Depth mismatch in Cyclone {i}"

print("All datasets have consistent depth levels")

All datasets have consistent spatial domain
All datasets have consistent depth levels


### Checking which cyclone belongs to which file

In [4]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

for i, ds in enumerate(cleaned_datasets):
    print(f"\n================ Cyclone {i+1} ================\n")
    
    print(ds)
    print("\nGlobal attributes:")
    print(ds.attrs)

    print("\nTime range:")
    print(ds.time.values[0], "→", ds.time.values[-1])


================ Cyclone 1 ================

<xarray.Dataset> Size: 143MB
Dimensions:    (time: 30, depth: 33, latitude: 60, longitude: 60)
Coordinates:
  * time       (time) datetime64[ns] 240B 2020-05-06 2020-05-07 ... 2020-06-04
  * depth      (depth) int16 66B 0 5 10 15 20 25 30 ... 400 450 500 550 600 700
  * latitude   (latitude) float32 240B 15.06 15.19 15.31 ... 22.19 22.31 22.44
  * longitude  (longitude) float32 240B 82.56 82.69 82.81 ... 89.69 89.81 89.94
Data variables:
    to         (time, depth, latitude, longitude) float64 29MB 30.56 ... nan
    so         (time, depth, latitude, longitude) float64 29MB 33.93 ... nan
    ugo        (time, depth, latitude, longitude) float64 29MB 0.821 ... nan
    vgo        (time, depth, latitude, longitude) float64 29MB 0.395 ... nan
    zo         (time, depth, latitude, longitude) float64 29MB 0.918 ... nan
Attributes:
    Conventions:               CF-1.0
    description:               ARMOR3D REP Copernicus Marine Service November

| Sl. No. | Cyclone Name | Year |
| ------: | ------------ | ---- |
|       1 | Amphan       | 2020 |
|       2 | Asani        | 2022 |
|       3 | Dana         | 2024 |
|       4 | Fani         | 2019 |
|       5 | Gulab        | 2021 |
|       6 | Hudhud       | 2014 |
|       7 | Phailin      | 2013 |
|       8 | Titli        | 2018 |
|       9 | Yaas         | 2021 |


### Saving Cleaned Datasets 

In [ ]:
cyclone_names = [
    "Amphan",
    "Asani",
    "Dana",
    "Fani",
    "Gulab",
    "Hudhud",
    "Phailin",
    "Titli",
    "Yaas"
]

for i, (ds, name) in enumerate(zip(cleaned_datasets, cyclone_names), start=1):
    ds = ds.copy()
    ds.attrs["cyclone_name"] = name
    ds.to_netcdf(f"cleaned_{i}_{name}.nc")